# AiFn: 16 lab keuangan dari data publik UCI 144

**Untuk pemula Python yang sudah mengenal Excel dan statistik dasar.** Jalankan setup, kemudian sel 1-16 berurutan. Tidak perlu memahami isi semua fungsi setup sebelum mulai: setiap lab menunjukkan input, operasi dan output. Python 3.10+ standard library, tanpa paket tambahan atau API.

Unduh `aifn-finance.csv` ke folder kerja yang sama. Jupyter lokal: Restart Kernel lalu Run All. Colab: unggah notebook dan CSV manual, kemudian Run All; Colab memerlukan jaringan, perhitungan lokal tidak.

Data: [Hofmann (1994), UCI Statlog German Credit](https://doi.org/10.24432/C5NC77), 1000 baris, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Perubahan: enam kolom sumber dipilih, label 1/2 dipetakan ke 0/1, ordinal sourceRow ditambah. Satuan amount tidak dinyatakan pada entri Attribute5 UCI, sehingga tetap unit sumber, bukan IDR. Duration bulan; age tahun; installmentRate tetap nilai asli 1-4. Label historis good/bad tidak membuktikan default 90 hari. Bukan data Indonesia, bukan nasihat atau keputusan kredit nyata.

**Ritme belajar setiap sesi:** kelas terpandu 30-45 menit (5 menit persoalan dan prediksi, 10 menit baca contoh output bersama, 15-20 menit jalankan dan cocokkan dengan Excel, 5-10 menit diskusi batas hasil). Tugas rumah: jalankan variasi yang diminta, tulis alasan dan simpan bukti output. Sel dibagikan untuk umum termasuk latihan UTS/UAS, bukan ujian resmi.

## Setup: baca CSV dan siapkan fungsi
**Alasan:** satu fungsi menyimpan rumus agar lab tidak menyalin perhitungan yang bisa berbeda. `def` mendefinisikan rumus, `for` mengulang untuk baris, `assert` memeriksa klaim. Output sel memastikan CSV yang benar; sel tidak mengunduh data.

In [ ]:
"""AiFn: 16 lab keuangan, Python 3 standard library, tanpa jaringan/API.
Simpan aifn-finance.csv di folder kerja. Jalankan sel berurutan atau python aifn-labs.py.
"""
import csv
import hashlib
import json
import math
from pathlib import Path
from statistics import mean, median
from typing import TypedDict

class Metrics(TypedDict):
    TP: int
    FP: int
    TN: int
    FN: int
    accuracy: float | None
    precision: float | None
    recall: float | None
    fpr: float | None

class ClusterResult(TypedDict):
    assignment: list[int]
    centers: list[list[float]]
    iterations: int
    converged: bool
    inertia: float
    history: list[float]
    normalization: tuple[list[float], list[float]]

DATA = Path('aifn-finance.csv')
if not DATA.is_file():
    raise FileNotFoundError('Unggah aifn-finance.csv ke folder kerja notebook, lalu Run All.')
FIELDS = ['sourceRow', 'duration', 'amount', 'installmentRate', 'existingCredits', 'age', 'risk']
with DATA.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == FIELDS, 'Header tidak sesuai; jangan ganti data tanpa audit.'
    rows = [{k: int(v) for k, v in r.items()} for r in reader]
assert len(rows) == 1000 and sum(r['risk'] for r in rows) == 300
assert len({r['sourceRow'] for r in rows}) == len(rows)
assert all(r['risk'] in (0, 1) and all(r[k] > 0 for k in FIELDS if k != 'risk') for r in rows)
FEATURES = ['duration', 'log1p(amount)', 'installmentRate', 'existingCredits']
REG_FEATURES = ['duration', 'installmentRate', 'existingCredits']

def split_records(items, seed=2026):
    state = seed & 0xffffffff
    train, test = [], []
    for label in (0, 1):
        group = [r for r in items if r['risk'] == label]
        for i in range(len(group)-1, 0, -1):
            state = (1664525*state + 1013904223) & 0xffffffff
            j = int(state / 4294967296 * (i+1))
            group[i], group[j] = group[j], group[i]
        n = int(len(group)*.8)
        train.extend(group[:n])
        test.extend(group[n:])
    return train, test

def vector(r):
    return [r['duration'], math.log1p(r['amount']), r['installmentRate'], r['existingCredits']]

def scaler(vectors):
    means = [mean(v[j] for v in vectors) for j in range(len(vectors[0]))]
    scales = [math.sqrt(mean((v[j]-m)**2 for v in vectors)) or 1 for j, m in enumerate(means)]
    return means, scales

def normalize(v, norm):
    return [(a-m)/s for a, m, s in zip(v, *norm)]

def dot(a, b):
    return sum(x*y for x, y in zip(a, b))

def sigmoid(z):
    return 1/(1+math.exp(-max(-40, min(40, z))))

def fit_logistic(items, iterations=600):
    norm = scaler([vector(r) for r in items])
    x = [[1]+normalize(vector(r), norm) for r in items]
    bad = sum(r['risk'] for r in items)
    w = [math.log((bad+.5)/(len(items)-bad+.5)), 0., 0., 0., 0.]
    for _ in range(iterations):
        gradient = [0.]*5
        for v, r in zip(x, items):
            error = sigmoid(dot(w, v))-r['risk']
            for j in range(5):
                gradient[j] += error*v[j]
        w = [a-.15*(g/len(items)+(.01*a if j else 0)) for j, (a, g) in enumerate(zip(w, gradient))]
    return {'weights': w, 'normalization': norm}

def predict(model, r):
    return sigmoid(dot(model['weights'], [1]+normalize(vector(r), model['normalization'])))

def scores_for(model, items):
    return [dict(score=predict(model, r), actual=r['risk'], age=r['age'], amount=r['amount'], duration=r['duration']) for r in items]

def ratio(a, b):
    return a/b if b else None

def metrics_at(scores, threshold) -> Metrics:
    m = dict(TP=0, FP=0, TN=0, FN=0)
    for r in scores:
        key = ('TP' if r['actual'] else 'FP') if r['score'] >= threshold else ('FN' if r['actual'] else 'TN')
        m[key] += 1
    return {'TP': m['TP'], 'FP': m['FP'], 'TN': m['TN'], 'FN': m['FN'], 'accuracy': ratio(m['TP']+m['TN'], len(scores)), 'precision': ratio(m['TP'], m['TP']+m['FP']), 'recall': ratio(m['TP'], m['TP']+m['FN']), 'fpr': ratio(m['FP'], m['FP']+m['TN'])}

def auc(scores):
    positive = [r['score'] for r in scores if r['actual']]
    negative = [r['score'] for r in scores if not r['actual']]
    return ratio(sum(1 if p > n else .5 if p == n else 0 for p in positive for n in negative), len(positive)*len(negative))

def solve(matrix, target):
    a = [list(row)+[y] for row, y in zip(matrix, target)]
    n = len(target)
    for j in range(n):
        pivot = max(range(j, n), key=lambda i: abs(a[i][j]))
        a[j], a[pivot] = a[pivot], a[j]
        assert abs(a[j][j]) > 1e-14
        divisor = a[j][j]
        a[j] = [x/divisor for x in a[j]]
        for i in range(n):
            if i != j:
                factor = a[i][j]
                a[i] = [x-factor*y for x, y in zip(a[i], a[j])]
    return [row[n] for row in a]

def fit_regression(items):
    vectors = [[r[f] for f in REG_FEATURES] for r in items]
    norm = scaler(vectors)
    x = [[1]+normalize(v, norm) for v in vectors]
    gram = [[sum(v[j]*v[k] for v in x)+(1e-8 if j == k and j else 0) for k in range(4)] for j in range(4)]
    rhs = [sum(v[j]*r['amount'] for v, r in zip(x, items)) for j in range(4)]
    weights = solve(gram, rhs)
    coefficients = [w/s for w, s in zip(weights[1:], norm[1])]
    intercept = weights[0]-dot(coefficients, norm[0])
    return intercept, coefficients

def reg_predict(model, r):
    return model[0]+dot(model[1], [r[f] for f in REG_FEATURES])

def kmeans(items, k=3, seed=2026) -> ClusterResult:
    norm = scaler([[r['amount'], r['duration']] for r in items])
    x = [normalize([r['amount'], r['duration']], norm) for r in items]
    def distance(a, b):
        return sum((v-w)**2 for v, w in zip(a, b))
    state = (1664525*(seed & 0xffffffff)+1013904223) & 0xffffffff
    centers = [x[int(state/4294967296*len(x))][:]]
    while len(centers) < k:
        distances = [min(distance(v, c) for c in centers) for v in x]
        centers.append(x[distances.index(max(distances))][:])
    assignment, history = [], []
    converged = False
    iteration = 0
    for iteration in range(1, 101):
        new = [min(range(k), key=lambda j: distance(v, centers[j])) for v in x]
        if new == assignment:
            converged = True
            break
        assignment = new
        for j in range(k):
            members = [v for v, a in zip(x, assignment) if a == j]
            if members:
                centers[j] = [mean(v[d] for v in members) for d in range(2)]
        history.append(sum(distance(v, centers[a]) for v, a in zip(x, assignment)))
    return {'assignment': assignment, 'centers': centers, 'iterations': iteration, 'converged': converged, 'inertia': history[-1], 'history': history, 'normalization': norm}

print('Setup:', len(rows), 'baris UCI 144; CSV SHA256:', hashlib.sha256(DATA.read_bytes()).hexdigest())



## Lab 1: Pengantar AI dan machine learning di keuangan

**Capaian RPS:** Menjelaskan peran AI dalam keuangan.

**Mengapa sel ini dijalankan:** Bedakan unit observasi, target dan tindakan. COUNTIF(risk,1) di Excel setara menghitung kondisi di Python. Label bad bukan bukti default dalam horizon tertentu.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Hitung proporsi good/bad dan tulis dua pertanyaan bisnis yang TIDAK dapat dijawab dataset ini.


In [ ]:
# LAB 1
problem = dict(unit='baris historis kredit Jerman', target='label good/bad sumber, bukan default 90 hari', purpose='belajar review risiko, bukan keputusan kredit nyata')
counts = {label: sum(r['risk'] == label for r in rows) for label in (0, 1)}
assert counts == {0: 700, 1: 300}
print('Lab 1: target dan distribusi aktual:', problem, counts)



## Lab 2: Python, Jupyter, dan pemrosesan data keuangan

**Capaian RPS:** Menyiapkan lingkungan Python dan data.

**Mengapa sel ini dijalankan:** Satu dict adalah satu baris Excel; daftar rows adalah tabel. r['amount'] memilih kolom. mean(...) setara AVERAGE. Jangan ubah nilai atau satuan untuk mempercantik hasil.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Cetak tiga baris pertama, hitung MIN/MAX duration, dan jelaskan mengapa sourceRow bukan nomor nasabah.


In [ ]:
# LAB 2
first = rows[0]
# Excel AVERAGE menjadi mean(generator); nama kolom menggantikan alamat sel.
average_amount = mean(r['amount'] for r in rows)
assert first['sourceRow'] == 1 and first['duration'] == 6 and first['amount'] == 1169
print('Lab 2: satu dict seperti satu baris Excel:', first, 'AVERAGE(amount)=', average_amount)



## Lab 3: Eksplorasi dan praproses data keuangan

**Capaian RPS:** Mengeksplorasi data keuangan.

**Mengapa sel ini dijalankan:** Histogram dan median memperlihatkan sebaran kredit yang tidak simetris. Train untuk belajar, test untuk ujian model. Parameter standardisasi berikutnya hanya dihitung dari train. Tidak ada tanggal untuk split waktu.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Buat histogram yang sama memakai COUNTIFS di Excel. Bandingkan mean dan median, lalu jelaskan dampak nilai ekstrem.


In [ ]:
# LAB 3
train, test = split_records(rows)
assert len(train) == 800 and len(test) == 200
assert not {r['sourceRow'] for r in train} & {r['sourceRow'] for r in test}
quantiles = [sorted(r['amount'] for r in rows)[int((len(rows)-1)*q)] for q in (.25, .5, .75)]
histogram = [sum(low <= r['amount'] < high for r in rows) for low, high in [(0, 2000), (2000, 5000), (5000, 10000), (10000, math.inf)]]
assert sum(histogram) == 1000
print('Lab 3: amount mean/median/quartiles', average_amount, median(r['amount'] for r in rows), quantiles, 'histogram', histogram, 'split', len(train), len(test))



## Lab 4: Supervised learning: regresi dan klasifikasi

**Capaian RPS:** Menjelaskan supervised learning.

**Mengapa sel ini dijalankan:** Target angka amount memerlukan regresi dan MAE; target kelas risk memerlukan klasifikasi. Baseline mayoritas dapat mencapai 70% tanpa mempelajari siapa yang berisiko.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Buktikan akurasi 70% dari COUNTIF. Jelaskan kerugian jika semua pengajuan dianggap good.


In [ ]:
# LAB 4
majority = int(mean(r['risk'] for r in train) >= .5)
baseline_accuracy = mean(r['risk'] == majority for r in test)
baseline_amount = mean(r['amount'] for r in train)
baseline_mae = mean(abs(r['amount']-baseline_amount) for r in test)
assert majority == 0 and baseline_accuracy == .7
print('Lab 4: klasifikasi mayoritas accuracy=', baseline_accuracy, 'regresi rata-rata train MAE=', baseline_mae)



## Lab 5: Model regresi untuk prediksi nilai

**Capaian RPS:** Membangun model regresi.

**Mengapa sel ini dijalankan:** Regresi mempelajari koefisien dari data train melalui persamaan linear dengan ridge sangat kecil untuk kestabilan. Target adalah amount aktual; amount tidak masuk input. MAE tetap dalam unit amount sumber, bukan rupiah.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Cocokkan satu prediksi dengan intercept + SUMPRODUCT(fitur,koefisien) di Excel. Jelaskan mengapa prediksi amount bukan rekomendasi limit.


In [ ]:
# LAB 5
regression = fit_regression(train)
regression_rows = [dict(actual=r['amount'], predicted=reg_predict(regression, r), duration=r['duration']) for r in test]
mae = mean(abs(r['actual']-r['predicted']) for r in regression_rows)
assert 'amount' not in REG_FEATURES and mae < baseline_mae
print('Lab 5: actual amount, tanpa amount sebagai input; intercept/coef=', regression, 'holdout MAE=', mae, 'baseline=', baseline_mae, 'contoh', regression_rows[:3])



## Lab 6: Model klasifikasi untuk keputusan keuangan

**Capaian RPS:** Membangun model klasifikasi.

**Mengapa sel ini dijalankan:** Logistic regression mengubah kombinasi fitur menjadi skor 0-1. log1p mengecilkan rentang amount, standardisasi memakai train. Loop 600 kali memperbaiki bobot terhadap label aktual, bukan rumus skor buatan.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Ubah jumlah iterasi pada salinan model train menjadi 50. Bandingkan train loss dan validation, jangan mengutak-atik final test untuk memilih model.


In [ ]:
# LAB 6
model = fit_logistic(train)
scores = scores_for(model, test)
train_scores = scores_for(model, train)
loss = -mean(r['actual']*math.log(r['score'])+(1-r['actual'])*math.log(1-r['score']) for r in train_scores)
prevalence = mean(r['risk'] for r in train)
baseline_loss = -prevalence*math.log(prevalence)-(1-prevalence)*math.log(1-prevalence)
assert loss < baseline_loss and len(scores) == 200 and all(0 < r['score'] < 1 for r in scores)
assert 'age' not in FEATURES
print('Lab 6: logistic 600 update; train log loss=', loss, 'baseline=', baseline_loss, 'holdout', metrics_at(scores, .5))



## Lab 7: Model credit scoring

**Capaian RPS:** Membangun model credit scoring.

**Mengapa sel ini dijalankan:** Ambang menentukan trade-off review manusia. FN=5 dan FP=1 mengikuti matriks biaya ilustratif sumber, tanpa mata uang. Pilih ambang memakai validation di dalam train, lalu laporkan test sekali.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Hitung biaya satu ambang dengan 5*FN+FP di Excel. Jelaskan mengapa biaya terendah belum menjamin kebijakan kredit layak.


In [ ]:
# LAB 7
# Pilih ambang pada split INTERNAL train; test final tidak dipakai memilih kebijakan.
inner_train, validation = split_records(train, 77)
inner_model = fit_logistic(inner_train)
validation_scores = scores_for(inner_model, validation)
thresholds = [.2, .35, .5, .65, .8]
validation_costs = {t: 5*metrics_at(validation_scores, t)['FN']+metrics_at(validation_scores, t)['FP'] for t in thresholds}
chosen = min(thresholds, key=lambda t: (validation_costs[t], t))
assert len(inner_train)+len(validation) == len(train)
print('Lab 7: biaya hipotetis tanpa mata uang (FN=5, FP=1), validation=', validation_costs, 'ambang=', chosen, 'test terkunci', metrics_at(scores, chosen))



## Lab 8: Ujian Tengah Semester (UTS)

**Capaian RPS:** Evaluasi capaian paruh pertama semester

**Mengapa sel ini dijalankan:** Ini latihan UTS publik, bukan soal rahasia atau nilai resmi. Assertion adalah pemeriksaan otomatis: bila klaim salah Python berhenti. Hubungkan setiap pemeriksaan dengan baseline atau leakage.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Tulis jawaban singkat: kapan standardisasi seluruh data bocor? Mengapa amount tidak boleh menjadi input prediksi amount? Sertakan angka MAE.


In [ ]:
# LAB 8
# Latihan UTS publik: cek klaim dengan angka, bukan hafalan istilah.
assert math.isclose(baseline_accuracy, .7)
assert len({r['sourceRow'] for r in train+test}) == 1000
assert mae < baseline_mae and sum(metrics_at(scores, .5)[k] for k in ('TP', 'FP', 'TN', 'FN')) == 200
print('Lab 8: latihan UTS publik lulus: split tidak tumpang tindih, baseline 70%, MAE membaik; bukan ujian resmi.')



## Lab 9: Analisis sentimen berita/keuangan dengan Claude

**Capaian RPS:** Menerapkan analisis sentimen.

**Mengapa sel ini dijalankan:** Tiga teks dan label referensi dinyatakan SINTETIS. Pencocokan kata gagal memahami laba vs biaya, negasi, dan target. Claude hanya opsional melalui antarmuka manual; tidak ada output Claude yang diklaim di sini.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Anotasi sendiri tiga teks dengan bukti frasa. Bila memakai Claude manual, simpan jawaban aktual dan bandingkan tanpa menyebut label sintetis sebagai ground truth nyata.


In [ ]:
# LAB 9
# Teks dan label referensi SINTETIS, bukan berita, anotasi manusia, atau keluaran Claude.
texts = [dict(text='Laba naik tetapi biaya naik', reference='perlu konteks'), dict(text='Kerugian turun, kas naik', reference='positif'), dict(text='Target laba naik; hasil belum tersedia', reference='perlu konteks')]
lexicon = {'naik': 1, 'turun': -1}
lexicon_results = [sum(lexicon.get(w.strip(';,').lower(), 0) for w in r['text'].split()) for r in texts]
assert len(lexicon_results) == len(texts)
print('Lab 9: teks dan label referensi SINTETIS; skor kata=', lexicon_results)
print('Claude MANUAL opsional, tanpa API: minta positif/negatif/perlu konteks, kutip bukti, pisahkan target dari hasil. Isi kolom jawaban sendiri; tidak ada keluaran Claude yang disediakan.')



## Lab 10: Evaluasi model: akurasi, presisi, dan AUC

**Capaian RPS:** Mengevaluasi performa model.

**Mengapa sel ini dijalankan:** Seluruh confusion matrix, ROC dan AUC berasal dari 200 skor logistic holdout yang sama. ROC mengubah ambang; AUC mengukur peringkat, bukan akurasi atau validasi probabilitas.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Rekonstruksi TP/FP/TN/FN di Excel untuk ambang .5. Pilih tiga titik ROC dan jelaskan mengapa AUC tidak berubah setelah transformasi skor monoton.


In [ ]:
# LAB 10
roc = [dict(threshold=t, **metrics_at(scores, t)) for t in [1.000001]+sorted({r['score'] for r in scores}, reverse=True)]
area = auc(scores)
def roc_area(points):
    total = 0.
    for a, b in zip(points, points[1:]):
        ax, ay, bx, by = a['fpr'], a['recall'], b['fpr'], b['recall']
        assert ax is not None and ay is not None and bx is not None and by is not None
        total += (bx-ax)*(by+ay)/2
    return total
trapezoid = roc_area(roc)
assert area is not None
assert math.isclose(area, trapezoid, abs_tol=1e-12)
assert auc([dict(r, score=3*r['score']+2) for r in scores]) == area
print('Lab 10: ROC dari 200 skor model yang sama, AUC=', area, 'trapezoid=', trapezoid, 'threshold .5', metrics_at(scores, .5))



## Lab 11: Interpretasi dan penjelasan model

**Capaian RPS:** Menginterpretasikan hasil model.

**Mengapa sel ini dijalankan:** Kontribusi fitur menjumlah ke log-odds dengan intercept, lalu sigmoid memberi skor. Ini penjelasan model terlatih untuk satu baris, bukan efek kausal dari mengubah durasi.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Hitung kontribusi duration satu baris secara manual. Jelaskan mengapa koefisien log1p(amount) tidak boleh dibaca sebagai rupiah.


In [ ]:
# LAB 11
example = test[0]
z_features = normalize(vector(example), model['normalization'])
contributions = {f: w*x for f, w, x in zip(FEATURES, model['weights'][1:], z_features)}
logit = model['weights'][0]+sum(contributions.values())
assert math.isclose(sigmoid(logit), predict(model, example))
print('Lab 11: kontribusi log-odds untuk sourceRow', example['sourceRow'], contributions, 'intercept', model['weights'][0], 'score', sigmoid(logit), 'bukan sebab-akibat atau PD terkalibrasi')



## Lab 12: Etika, bias, dan responsible AI di keuangan

**Capaian RPS:** Menilai etika dan bias AI.

**Mengapa sel ini dijalankan:** Age tidak masuk model tetapi bisa berkaitan dengan fitur lain. FPR kelompok dibagi jumlah good di kelompok, bukan jumlah semua anggota. Cutoff 25 hanya pilihan latihan; sampel kecil dan label historis membatasi kesimpulan.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Laporkan denominator good, FP dan FPR tiap kelompok. Tulis apa yang masih perlu diaudit sebelum menyatakan model adil.


In [ ]:
# LAB 12
audit = []
for name, predicate in [('age < 25', lambda r: r['age'] < 25), ('age >= 25', lambda r: r['age'] >= 25)]:
    members = [r for r in scores if predicate(r)]
    m = metrics_at(members, chosen)
    audit.append(dict(group=name, count=len(members), good=m['FP']+m['TN'], bad=m['TP']+m['FN'], **m))
assert sum(g['count'] for g in audit) == 200
assert metrics_at([], .5)['fpr'] is None
print('Lab 12: cutoff usia 25 adalah pilihan latihan, bukan standar hukum;', audit, 'FPR=None bila denominator good nol. Tanpa age pada fitur bukan jaminan adil.')



## Lab 13: Unsupervised learning: clustering untuk segmentasi nasabah

**Capaian RPS:** Menerapkan clustering untuk segmentasi.

**Mengapa sel ini dijalankan:** K-means menstandarkan amount dan duration agar satu unit tidak mendominasi. Assign dan update centroid diulang sampai anggota stabil. Inertia turun, tetapi cluster bukan kelas good/bad.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Bandingkan centroid dengan AVERAGEIFS pada cluster. Jelaskan kebutuhan layanan yang mungkin berbeda tanpa menganggap cluster sebagai risiko kredit.


In [ ]:
# LAB 13
clusters = kmeans(rows)
assert clusters['converged'] and 1 < clusters['iterations'] < 100
assert all(b <= a+1e-8 for a, b in zip(clusters['history'], clusters['history'][1:]))
segments = [dict(cluster=j, count=clusters['assignment'].count(j), amount=mean(r['amount'] for r, a in zip(rows, clusters['assignment']) if a == j), duration=mean(r['duration'] for r, a in zip(rows, clusters['assignment']) if a == j)) for j in range(3)]
assert sum(s['count'] for s in segments) == 1000
print('Lab 13: k-means amount/duration distandardisasi, iterasi=', clusters['iterations'], 'inertia=', clusters['inertia'], 'segmen=', segments, 'nomor cluster bukan kelas risiko')



## Lab 14: Penyempurnaan model dan dokumentasi

**Capaian RPS:** Menyempurnakan dan mendokumentasikan model.

**Mengapa sel ini dijalankan:** Eksperimen memakai pembagian internal train agar test tidak menjadi leaderboard. Catat seed, sumber, fitur, metrik dan batas pakai pada model card. Perbedaan validation AUC bukan bukti keunggulan universal.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Tambahkan satu seed validation dan dokumentasikan hasil termasuk kegagalan. Tetapkan pilihan sebelum melihat final test lagi.


In [ ]:
# LAB 14
# Eksperimen seed di data TRAIN saja; test final tidak menjadi leaderboard tuning.
experiments = []
for seed in (17, 42):
    fit_rows, val_rows = split_records(train, seed)
    candidate = fit_logistic(fit_rows, iterations=300)
    experiments.append(dict(seed=seed, validationCount=len(val_rows), auc=auc(scores_for(candidate, val_rows))))
assert all(e['validationCount'] == 160 for e in experiments)
model_card = dict(source='https://doi.org/10.24432/C5NC77', csv_sha256=hashlib.sha256(DATA.read_bytes()).hexdigest(), features=FEATURES, seed=2026, train=800, test=200, threshold=chosen, threshold_selected_on='inner train validation', test_auc=area, test_mae=mae, limitations=['historical selected German sample', 'no dates or verified default horizon', 'no external validation or probability calibration', 'not an Indonesia lending model'])
print('Lab 14: eksperimen validation', experiments, 'model card', json.dumps(model_card, ensure_ascii=False))



## Lab 15: Proyek: credit-scoring atau sentiment model

**Capaian RPS:** Mempresentasikan model AI keuangan.

**Mengapa sel ini dijalankan:** Presentasi proyek perlu pertanyaan, data, baseline, denominator, hasil dan keterbatasan. Angka pada evidence dihitung dari model, bukan diketik untuk mengesankan audiens. Jalur sentimen memerlukan anotasi manual dan sumber teks yang sah.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Susun lima slide atau lima paragraf: masalah, sumber, metode, hasil vs baseline, risiko dan keputusan. Sertakan CSV hash dan larangan penggunaan nyata.


In [ ]:
# LAB 15
final_metrics = metrics_at(scores, chosen)
presentation = dict(question=problem['target'], evidence=dict(n=200, matrix={k: final_metrics[k] for k in ('TP', 'FP', 'TN', 'FN')}, auc=area, mae=mae), proposed_action='review manusia dalam simulasi', limitation=model_card['limitations'])
assert sum(presentation['evidence']['matrix'].values()) == presentation['evidence']['n']
print('Lab 15: bukti presentasi proyek credit-scoring=', json.dumps(presentation, ensure_ascii=False))



## Lab 16: Ujian Akhir Semester (UAS)

**Capaian RPS:** Evaluasi capaian menyeluruh satu semester

**Mengapa sel ini dijalankan:** Latihan UAS publik menguji keterulangan end-to-end. Nilai rujukan numerik mengunci kesetaraan JS/Python, bukan target performa yang boleh dikejar dengan mengganti data.

**Kerjakan di kelas:** prediksi bentuk output, jalankan sel, baca setiap angka bersama pasangan, lalu jelaskan dengan bahasa keuangan.

**Tugas notebook di rumah:** Restart Kernel dan Run All. Jelaskan tiga keterbatasan, satu trade-off biaya dan satu temuan audit dengan denominator; lampirkan model card. Tidak ada penilaian resmi otomatis.


In [ ]:
# LAB 16
assert sum(counts.values()) == 1000 and len(scores) == 200
assert set(FEATURES).isdisjoint({'risk', 'age', 'sourceRow'})
assert clusters['converged'] and model_card['threshold_selected_on'] == 'inner train validation'
assert math.isclose(area, .6536904761904762, abs_tol=1e-12)
assert math.isclose(mae, 1466.1160154055108, abs_tol=1e-7)
print('Lab 16: latihan UAS publik: model, audit, sumber, baseline dan batas penggunaan terverifikasi; bukan ujian resmi.')
print('CROSS_LANGUAGE', json.dumps(dict(auc=area, mae=mae, baselineMae=baseline_mae, weights=model['weights'], firstScore=scores[0]['score'], inertia=clusters['inertia'], clusterIterations=clusters['iterations'])))
print('ALL OFFLINE LAB ASSERTIONS PASSED')
